# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all RecordSets and their fields by @id
record_sets = list(dataset.metadata.recordSets)
print(f"Number of record sets: {len(record_sets)}\n")

for idx, rs in enumerate(record_sets):
    print(f"Record set {idx+1}: @id = {rs.id}, name = {getattr(rs, 'name', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"      - @id = {field.id}, name = {getattr(field, 'name', None)}")
    print()

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. Use the record sets and fields `@id`s from the overview above.

In [ ]:
# Extract all data from all record sets into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]

# Load the records (data) for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: '{record_set_id}' with {len(df)} rows and {len(df.columns)} columns.")

# Display the columns of the first record set
selected_record_set_id = record_set_ids[0]
print(f"\nColumns in record set '{selected_record_set_id}':\n{dataframes[selected_record_set_id].columns.tolist()}")
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping/categorizing data.

*You must use field `@id`s and names from section 2 to set variables below for your tasks.*

In [ ]:
# Choose a numeric field (update ID below if desired):
record_set_id = selected_record_set_id
df = dataframes[record_set_id]

# Discover likely numeric fields
print("\nNumeric candidate fields in this record set:")
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(numeric_candidates)

# Let's pick one for demonstration (if none, code won't error):
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    threshold = df[numeric_field].mean() # for demo, filter above average
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a candidate categorical field
    group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    print("\nGroup-by candidates:", group_candidates)
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped mean of numeric fields by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric fields detected in this record set.")

## 5. Visualization
Visualize a numeric field's distribution and the relationship between two fields.

*These examples use matplotlib and seaborn for convenience.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group/categoric variable exists, boxplot
    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`, listed its record sets and fields by `@id`, and performed basic data extraction, analysis, and visualization. 

Next steps could involve deeper EDA, domain-specific analysis, or preparing the data for ML tasks.